# Final Project Notebook Plan - Group 2: Minh Phuc Nguyen, Zaid Fafouri, Takumi Kojima

This notebook documents all code and results completed so far, in two training stages:

1. Fine-tune `yolo26m`
2. Fine-tune `rtdetr-l`

## 1) Goal

- Fine tune and compare two object detection models on the same dataset split and evaluation flow.
- Keep training setup reproducible with explicit config and run names.
- Collect key outputs: training logs, best weights, validation metrics, and sample predictions.

## 2) Setup

In this step, we prepare the runtime and dataset before training:

- Install all required packages directly in one `pip install` code cell
- Download the BDD100K dataset
- Organize data into the expected project structure (`data/`)
- Verify the training YAML and class mapping (`data/yolo_3cls/bdd100k_3cls.yaml`)

Output of this step:
- Environment is ready
- BDD100K is available for training and inference


In [1]:
pip install certifi charset-normalizer contourpy cuda-bindings cuda-pathfinder cycler filelock fonttools fsspec idna Jinja2 kiwisolver MarkupSafe matplotlib mpmath networkx numpy nvidia-cublas-cu12 nvidia-cuda-cupti-cu12 nvidia-cuda-nvrtc-cu12 nvidia-cuda-runtime-cu12 nvidia-cudnn-cu12 nvidia-cufft-cu12 nvidia-cufile-cu12 nvidia-curand-cu12 nvidia-cusolver-cu12 nvidia-cusparse-cu12 nvidia-cusparselt-cu12 nvidia-nccl-cu12 nvidia-nvjitlink-cu12 nvidia-nvshmem-cu12 nvidia-nvtx-cu12 opencv-python packaging pillow polars polars-runtime-32 psutil pyparsing python-dateutil PyYAML requests scipy setuptools six sympy torch torchvision triton typing_extensions ultralytics ultralytics-thop urllib3 kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 3.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.6/362.6 kB 5.1 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 10.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━

## 3) Download BDD100K and Prepare Data

This step downloads BDD100K into `data/`, converts raw JSON annotations to YOLO 3-class format, and validates training-ready outputs.


In [2]:
%pip install -q kagglehub
from pathlib import Path
import tarfile
import zipfile
import kagglehub

download_path = Path(kagglehub.dataset_download('solesensei/solesensei_bdd100k'))
print('Downloaded path:', download_path)

Note: you may need to restart the kernel to use updated packages.


/home/phuc/Work/CSC_331/CSC331-Final/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 7.61G/7.61G [14:08<00:00, 9.62MB/s]

Extracting files...


Downloaded path: /home/phuc/.cache/kagglehub/datasets/solesensei/solesensei_bdd100k/versions/2


In [3]:
from pathlib import Path
import shutil
import json

cache_root = Path.home() / ".cache" / "kagglehub" / "datasets" / "solesensei" / "solesensei_bdd100k" / "versions" / "2"
images_src = cache_root / "bdd100k" / "bdd100k" / "images" / "100k"
labels_src = cache_root / "bdd100k_labels_release" / "bdd100k" / "labels"

images_dst = Path("data/images")
labels_dst = Path("data/labels")

splits = ("train", "val", "test")
img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Clean and recreate destination splits
for s in splits:
    if (images_dst / s).exists():
        shutil.rmtree(images_dst / s)
    if (labels_dst / s).exists():
        shutil.rmtree(labels_dst / s)
    (images_dst / s).mkdir(parents=True, exist_ok=True)
    (labels_dst / s).mkdir(parents=True, exist_ok=True)

# 1) Copy images recursively (handles nested trainA/trainB/testA/testB)
for split in splits:
    src_split = images_src / split
    if not src_split.exists():
        raise FileNotFoundError(f"Missing source image split: {src_split}")

    copied = 0
    for p in src_split.rglob("*"):
        if p.is_file() and p.suffix.lower() in img_exts:
            shutil.copy2(p, images_dst / split / p.name)
            copied += 1
    print(f"copied images {split}: {copied}")

# 2) Expand consolidated labels JSON -> per-image JSON files
label_map = {
    "train": labels_src / "bdd100k_labels_images_train.json",
    "val": labels_src / "bdd100k_labels_images_val.json",
}

for split, src_json in label_map.items():
    if not src_json.exists():
        raise FileNotFoundError(f"Missing labels json: {src_json}")

    data = json.loads(src_json.read_text(encoding="utf-8"))
    if not isinstance(data, list):
        raise RuntimeError(f"Unexpected JSON format in {src_json}")

    out_dir = labels_dst / split
    for item in data:
        name = item.get("name")
        if not name:
            continue
        (out_dir / f"{name}.json").write_text(json.dumps(item), encoding="utf-8")

# Note: no official test detection labels in this release
print("labels test: 0 (not provided in this dataset release)")

# 3) Print final counts in data/
def count_images(d: Path) -> int:
    return sum(1 for p in d.rglob("*") if p.is_file() and p.suffix.lower() in img_exts)

for split in splits:
    img_n = count_images(images_dst / split)
    lbl_n = len(list((labels_dst / split).glob("*.json")))
    print(f"data/{split}: images={img_n}, labels_json={lbl_n}")


copied images train: 70000
copied images val: 10000
copied images test: 20000
labels test: 0 (not provided in this dataset release)
data/train: images=70000, labels_json=69863
data/val: images=10000, labels_json=10000
data/test: images=20000, labels_json=0


In [4]:
from pathlib import Path
import json
import shutil
from PIL import Image

# Paths
data_dir = Path("data")
images_src = data_dir / "images"
labels_src = data_dir / "labels"
out_dir = data_dir / "yolo_3cls"
images_out = out_dir / "images"
labels_out = out_dir / "labels"

# Config
splits = ("train", "val", "test")
class_map = {
    "traffic sign": 0,
    "person": 1,
    "car": 2,
    "truck": 2,
    "bus": 2,
    "train": 2,
    "motorcycle": 2,
    "bicycle": 2,
}
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def clip_box(x1, y1, x2, y2, w, h):
    x1 = min(max(float(x1), 0.0), float(w))
    y1 = min(max(float(y1), 0.0), float(h))
    x2 = min(max(float(x2), 0.0), float(w))
    y2 = min(max(float(y2), 0.0), float(h))
    return x1, y1, x2, y2

def to_yolo_line(cid, x1, y1, x2, y2, w, h):
    bw = x2 - x1
    bh = y2 - y1
    xc = x1 + bw / 2.0
    yc = y1 + bh / 2.0
    return f"{cid} {xc/w:.6f} {yc/h:.6f} {bw/w:.6f} {bh/h:.6f}"

# Recreate output folders
if out_dir.exists():
    shutil.rmtree(out_dir)
(images_out).mkdir(parents=True, exist_ok=True)
(labels_out).mkdir(parents=True, exist_ok=True)

# Copy images and convert labels
for split in splits:
    src_img_split = images_src / split
    src_lbl_split = labels_src / split
    dst_img_split = images_out / split
    dst_lbl_split = labels_out / split
    dst_img_split.mkdir(parents=True, exist_ok=True)
    dst_lbl_split.mkdir(parents=True, exist_ok=True)

    img_files = [p for p in src_img_split.rglob("*") if p.is_file() and p.suffix.lower() in image_exts]
    converted = 0

    for img_path in img_files:
        shutil.copy2(img_path, dst_img_split / img_path.name)
        stem = img_path.stem
        json_path = src_lbl_split / f"{stem}.json"
        txt_path = dst_lbl_split / f"{stem}.txt"

        lines = []
        if json_path.exists():
            try:
                ann = json.loads(json_path.read_text(encoding="utf-8"))
                objs = ann.get("frames", [{}])[0].get("objects", [])
                with Image.open(img_path) as im:
                    iw, ih = im.size

                for obj in objs:
                    cat = obj.get("category")
                    if cat not in class_map:
                        continue
                    box = obj.get("box2d")
                    if not isinstance(box, dict):
                        continue
                    try:
                        x1, y1, x2, y2 = clip_box(box["x1"], box["y1"], box["x2"], box["y2"], iw, ih)
                    except Exception:
                        continue
                    if x2 <= x1 or y2 <= y1:
                        continue
                    lines.append(to_yolo_line(class_map[cat], x1, y1, x2, y2, iw, ih))
            except Exception:
                pass

        txt_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
        converted += 1

    print(f"{split}: images={len(img_files)}, yolo_labels={converted}")

# Write YAML
yaml_path = out_dir / "bdd100k_3cls.yaml"
yaml_path.write_text(
    "path: data/yolo_3cls\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n"
    "names: [traffic_sign, pedestrian, vehicle]\n",
    encoding="utf-8",
)

print("YAML written:", yaml_path)


train: images=70000, yolo_labels=70000
val: images=10000, yolo_labels=10000
test: images=20000, yolo_labels=20000
YAML written: data/yolo_3cls/bdd100k_3cls.yaml


## 4) Train YOLO26m

In this step, we train the first detection model using the prepared YOLO-format dataset.

- Use dataset config: `data/yolo_3cls/bdd100k_3cls.yaml`
- Start with a short smoke run to verify training setup
- Run full fine-tuning for `yolo26m`
- Save outputs (weights, logs, metrics, and plots) for later comparison

Output of this step:
- Trained YOLO26m checkpoints (`best.pt`, `last.pt`)
- Training metrics and validation results
- Run folder ready for model comparison


In [5]:
import os
from pathlib import Path
import torch
from ultralytics import YOLO

# Avoid Ultralytics config warning in notebook runtimes
os.environ["YOLO_CONFIG_DIR"] = "/tmp/Ultralytics"

# Part 4: YOLO26m setup config
cfg = {
    "model": "yolo26m.pt",  # must exist in current working directory, or be downloadable by Ultralytics
    "data": "data/yolo_3cls/bdd100k_3cls.yaml",
    "device": "0,1",
    "batch": 4,  # RTX 5070ti 16GB x2
    "workers": 8,
    "optimizer": "AdamW",
    "lr0": 5e-4,
    "imgsz": 1280,
    "epochs": 15,
    "patience": 10,
    "cos_lr": True,
    "close_mosaic": 15,
    "mosaic": 0.0,
    "amp": True,
    "seed": 0,
    "val": True,
    "project": "runs/train",
    "name": "bdd3_y26m_1280_e15_ft_b128_h100x8",
    "exist_ok": False,
}

# Environment checks
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

requested = [x.strip() for x in cfg["device"].split(",") if x.strip()]
if torch.cuda.is_available() and len(requested) > torch.cuda.device_count():
    print(f"WARNING: requested {len(requested)} GPUs, but only {torch.cuda.device_count()} visible.")

# Dataset check
if not Path(cfg["data"]).exists():
    raise FileNotFoundError(f"Dataset YAML not found: {cfg['data']}")

# Model check (fail early with clear message)
try:
    _ = YOLO(cfg["model"])
    print(f"Model ready: {cfg['model']}")
except Exception as e:
    raise RuntimeError(
        f"Cannot load model '{cfg['model']}'. Put 'yolo26m.pt' in the current directory "
        f"or change cfg['model'] to a valid weight path/model name.\nOriginal error: {e}"
    )

print("\nTraining config ready:")
for k, v in cfg.items():
    print(f"  {k}: {v}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/phuc/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
Model ready: yolo26m.pt

Training config ready:
  model: yolo26m.pt
  data: data/yolo_3cls/bdd100k_3cls.yaml
  device: 0,1
  batch: 4
  workers: 8
  optimizer: AdamW
  lr0: 0.0005
  imgsz: 1280
  epochs: 15
  patience: 10
  cos_lr: True
  close_mosaic: 15
  mosaic: 0.0
  amp: True
  seed: 0
  val: True
  project: runs/train
  name: bdd3_y26m_1280_e15_ft_b128_h100x8
  exist_ok: False


In [6]:
from ultralytics import YOLO

# Smoke run: 1 epoch using the Part 4 config dict `cfg`
smoke_cfg = dict(cfg)
smoke_cfg["epochs"] = 1
smoke_cfg["name"] = f"{cfg['name']}_smoke"
smoke_cfg["exist_ok"] = True

model = YOLO(smoke_cfg["model"])
model.train(
    data=smoke_cfg["data"],
    epochs=smoke_cfg["epochs"],
    imgsz=smoke_cfg["imgsz"],
    batch=smoke_cfg["batch"],
    device=smoke_cfg["device"],
    project=smoke_cfg["project"],
    name=smoke_cfg["name"],
    exist_ok=smoke_cfg["exist_ok"],
    patience=smoke_cfg["patience"],
    val=smoke_cfg["val"],
    workers=smoke_cfg["workers"],
    optimizer=smoke_cfg["optimizer"],
    cos_lr=smoke_cfg["cos_lr"],
    close_mosaic=smoke_cfg["close_mosaic"],
    mosaic=smoke_cfg["mosaic"],
    amp=smoke_cfg["amp"],
    seed=smoke_cfg["seed"],
    lr0=smoke_cfg["lr0"],
)

print("Smoke run complete:", f"{smoke_cfg['project']}/{smoke_cfg['name']}")


Ultralytics 8.4.24 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 15818MiB)
                                                      CUDA:1 (NVIDIA GeForce RTX 5070 Ti, 15840MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=data/yolo_3cls/bdd100k_3cls.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, n

W0321 23:34:13.606000 19218 venv/lib/python3.12/site-packages/torch/distributed/elastic/agent/server/api.py:739] Received 2 death signal, shutting down workers
W0321 23:34:13.606000 19218 venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/api.py:1010] Sending process 19221 closing signal SIGINT
Process Process-18:
Process Process-1:
Process Process-14:
Process Process-17:
Process Process-20:
W0321 23:34:13.606000 19218 venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/api.py:1010] Sending process 19222 closing signal SIGINT
Process Process-20:
Process Process-11:
Process Process-10:
Process Process-5:
Process Process-13:
Process Process-7:
Process Process-15:
Process Process-7:
Process Process-6:
Process Process-19:
Process Process-3:
Process Process-12:
Process Process-19:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
  File "/usr/lib/pytho

KeyboardInterrupt: 

Exception ignored in: <Finalize object, dead>
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/util.py", line 227, in __call__
  File "/usr/lib/python3.12/multiprocessing/util.py", line 469, in close_fds
AttributeError: 'NoneType' object has no attribute 'close'
Exception ignored in: <Finalize object, dead>
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/util.py", line 227, in __call__
  File "/usr/lib/python3.12/multiprocessing/util.py", line 469, in close_fds
AttributeError: 'NoneType' object has no attribute 'close'
Exception ignored in: <Finalize object, dead>
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/util.py", line 227, in __call__
  File "/usr/lib/python3.12/multiprocessing/util.py", line 469, in close_fds
AttributeError: 'NoneType' object has no attribute 'close'
Exception ignored in: <Finalize object, dead>
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocess